# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveenadanthapally/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane: Refresh / Content Opportunity Scoring**

I frame this as a **scoring problem used to create a ranked review queue**.

The ideal outcome is to help a content team identify which pages should be reviewed first for a possible refresh. The model's goal would be to assign each content page a score representing its likelihood of showing signals associated with declining performance.

A ranked score is useful because the team has limited review capacity. Instead of treating every page the same, the output can prioritize the pages with the strongest evidence for review.

The final output supports a content action: **review the highest-scoring pages first and decide whether they need a refresh, improvement, protection, or monitoring.**


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


The ideal outcome, "identify pages that deserve a refresh," is not directly observed in the dataset because there is no column saying whether a page actually needed or benefited from a refresh.

I would therefore use a **proxy target** for the initial analysis:

`is_declining = (trend_direction == "down")`

This proxy identifies pages whose observed trend is downward in the available data. It is not a direct measurement of whether a refresh would improve the page, so I would describe the resulting model as **decision-support**, not as a causal prediction of refresh success.

The target would therefore be a binary indicator:

* `1` = observed downward trend
* `0` = not observed as downward

The dataset also contains `trend_pct`, which gives the magnitude of the observed change and could be useful for later analysis, but the initial framing keeps the proxy simple.



In [1]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/praveenadanthapally/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

df["is_declining"] = (df["trend_direction"] == "down").astype(int)

print("Rows:", len(df))

print("\nProxy target counts:")
print(df["is_declining"].value_counts(dropna=False))

print("\nProxy target rate:")
print(df["is_declining"].mean())

Rows: 30000

Proxy target counts:
is_declining
1    16262
0    13738
Name: count, dtype: int64

Proxy target rate:
0.5420666666666667


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The primary success metric for the ranked review queue would be **Precision@50**.

Precision@50 asks: among the 50 pages given the highest priority by the scoring system, what proportion actually have the declining proxy label?

This metric matches the real content action because a content team has limited review capacity. A useful system should put a high proportion of relevant pages near the top of the queue.

For the eventual ML system, I would consider higher Precision@50 better. I would also compare the model against a simple baseline so that improvement is measured rather than assumed.

The business/content success is not "the model is accurate." The practical success is that reviewers can find more relevant pages in their limited top-50 review queue.


In [2]:


baseline_precision_at_50 = df["is_declining"].mean()

print(f"Overall declining proxy rate: {baseline_precision_at_50:.3f}")
print(f"Expected random-selection Precision@50: {baseline_precision_at_50:.3f}")

Overall declining proxy rate: 0.542
Expected random-selection Precision@50: 0.542


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The **unit of analysis is one content page**.

Each row represents one `content_id`, along with its search, content, traffic, engagement, age, freshness, and trend-related attributes.

For example, `content_id` identifies the page, while fields such as `search_volume`, `content_age_days`, `impressions_90d`, `sessions_90d`, `ctr`, `avg_position`, and `trend_direction` describe that page.

The scoring system would therefore produce **one score per content page**. Those scores could then be sorted to create a prioritized content-review queue.


In [4]:
# Show the actual unit of analysis.
# One row represents one content page.

display(
    df[
        [
            "content_id",
            "client_id",
            "search_volume",
            "impressions_90d",
            "sessions_90d",
            "content_age_days",
            "days_since_last_update",
            "ctr",
            "avg_position",
            "trend_direction",
            "trend_pct",
            "is_declining",
        ]
    ].head(10)
)
print("Number of rows:", len(df))
print("Number of unique content pages:", df["content_id"].nunique())
print("One row per content page:", len(df) == df["content_id"].nunique())

,content_id,client_id,search_volume,impressions_90d,sessions_90d,content_age_days,days_since_last_update,ctr,avg_position,trend_direction,trend_pct,is_declining
0,content_304f48230142,client_f369cb89fc,10.0,3803,17,187,20,0.76,10.6,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,15320,9,445,25,0.05,20.3,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,12581,11,141,20,0.09,36.5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,11751,78,463,22,0.49,6.2,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,19140,145,263,14,0.13,44.0,down,-34.7,1
5,content_d4084a4bc775,client_f369cb89fc,720.0,3970,5,147,20,0.03,8.5,down,-38.9,1
6,content_9a34b442b552,client_8722616204,0.0,20,1,90,20,0.00,7.0,down,-92.3,1
7,content_a63219c6e95a,client_19581e27de,590.0,1724,28,445,22,0.06,21.2,stable,0.6,0
8,content_5e6c160719bc,client_6208ef0f77,0.0,32574,68,90,20,0.09,46.0,down,-58.8,1
9,content_c27558df2b0c,client_19581e27de,0.0,1240,3,257,104,0.16,4.9,down,-29.2,1


Number of rows: 30000
Number of unique content pages: 30000
One row per content page: True


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule could identify declining pages using one threshold, such as `trend_pct < -20`, but that would ignore the different signals available for each page.

The dataset contains multiple potentially useful signals, including search demand, recent and previous impressions, clicks, sessions, CTR, average position, content age, days since the last update, engagement, and trend information. These signals can interact in ways that are difficult to capture with one fixed threshold.

For example, a large traffic decline on a very old page may represent a different review priority from the same decline on a recently updated page. Similarly, a page with high search demand may deserve different attention from a page with almost no search demand.

ML could learn combinations of these signals from observed data and produce a continuous priority score rather than relying on one manually chosen cutoff.

However, I would still compare the ML approach against simple rules and baselines. ML is useful only if it produces a meaningfully better ranked review queue.


In [5]:
summary = df[
    [
        "search_volume",
        "impressions_90d",
        "sessions_90d",
        "content_age_days",
        "days_since_last_update",
        "ctr",
        "avg_position",
        "trend_pct",
    ]
].describe().T

display(summary)

,count,mean,std,min,25%,50%,75%,max
search_volume,27532.0,158.882391,1518.270825,0.0,0.0,10.00,20.00,74000.0
impressions_90d,30000.0,5200.366300,16838.019547,1.0,81.0,731.00,3615.25,517715.0
sessions_90d,30000.0,37.066633,107.069131,1.0,2.0,7.00,27.00,4345.0
content_age_days,30000.0,256.167800,132.707930,90.0,132.0,236.00,333.00,564.0
days_since_last_update,30000.0,46.098300,42.078709,1.0,20.0,20.00,104.00,373.0
ctr,30000.0,0.510733,3.279162,0.0,0.0,0.07,0.29,100.0
avg_position,30000.0,16.342380,15.216790,0.0,6.2,10.80,22.30,245.0
trend_pct,26612.0,-4.785969,473.861780,-100.0,-62.6,-33.50,0.00,44900.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.